# Exploración inicial de datos
### By: Maricel Karina Martínez Hernández

### Date: 2026-08-21

### Descripción:

Revisión y exploración general de los datos para verificar los tipos de datos y solucionar cualquier problema relacionado con ellos. Esto se realiza con el fin de llevar a cabo un correcto análisis y visualización de los datos.

### Requerimiento

Crear una nueva rama de git (Usar Gitflow) y Crear un notebook para la exploración inicial de los datos

Tomar como ejemplo los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/
el objetivo es realizar una exploración general de datos para verificar los tipos de datos con el fin de comprender de las características y el esquema de datos y si es posible solucionar problemas básicos relacionados. realizar:

Descripción general de los datos
Unificar la forma como se representan los valores Nulos
Convertir los datos en su tipo correcto (numéricos, categóricos, booleanos, fechas, etc) y corrección de los datos si es necesario, para eu cada columna tenga un tipo de dato uniforme.
Almacenar el dataset final en un formato adecuado como .parquet
### Entregables
Notebook con la exploración general de los datos y los pasos descritos anteriormente

Se debe realizar un Pull request para ingresar el notebook a la ramamain para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD


## 📚 Importar librerías

In [1]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

## 💾 Cargar datos

In [2]:
data_path = Path.cwd().resolve().parents[1] / "data" / "01_raw" / "Precios_Casas_Boston.csv"

boston_df = pd.read_csv(data_path, low_memory=False)
boston_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       448 non-null    int64  
 1   crim     443 non-null    float64
 2   zn       446 non-null    float64
 3   indus    442 non-null    float64
 4   chas     446 non-null    float64
 5   nox      444 non-null    float64
 6   rm       442 non-null    float64
 7   age      444 non-null    float64
 8   dis      442 non-null    float64
 9   rad      444 non-null    float64
 10  tax      443 non-null    float64
 11  ptratio  446 non-null    float64
 12  black    445 non-null    float64
 13  lstat    447 non-null    float64
 14  medv     432 non-null    float64
dtypes: float64(14), int64(1)
memory usage: 52.6 KB


## 📊 Descripción de los datos

In [3]:
boston_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       448 non-null    int64  
 1   crim     443 non-null    float64
 2   zn       446 non-null    float64
 3   indus    442 non-null    float64
 4   chas     446 non-null    float64
 5   nox      444 non-null    float64
 6   rm       442 non-null    float64
 7   age      444 non-null    float64
 8   dis      442 non-null    float64
 9   rad      444 non-null    float64
 10  tax      443 non-null    float64
 11  ptratio  446 non-null    float64
 12  black    445 non-null    float64
 13  lstat    447 non-null    float64
 14  medv     432 non-null    float64
dtypes: float64(14), int64(1)
memory usage: 52.6 KB


In [4]:
boston_df.sample(10, random_state=42)

,ID,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
285,442,9.72418,0.0,18.10,0.0,0.74,6406.00,97.2,2.0651,24.0,666.0,20.2,385.96,19.52,17.1
296,458,8.20058,0.0,18.10,0.0,713.00,5936.00,80.3,2.7792,24.0,666.0,20.2,3.50,16.94,13.5
117,172,2.31390,0.0,19.58,0.0,605.00,5.88,97.3,2.3887,5.0,403.0,14.7,348.13,12.03,19.1
346,433,6.44405,0.0,18.10,0.0,584.00,6425.00,74.8,2.2004,24.0,666.0,20.2,97.95,12.03,16.1
70,104,0.21161,0.0,8.56,0.0,0.52,6137.00,87.4,2.7147,5.0,384.0,20.9,394.47,13.44,19.3
30,50,0.21977,0.0,6.91,0.0,448.00,5602.00,62.0,6.0877,3.0,233.0,17.9,396.90,16.20,19.4
192,287,0.01965,80.0,1.76,0.0,385.00,6.23,31.5,9.0892,1.0,241.0,18.2,341.60,12.93,20.1
79,119,0.13058,0.0,10.01,0.0,547.00,5872.00,73.1,2.4775,6.0,432.0,17.8,338.63,15.37,20.4
406,129,0.32543,0.0,21.89,0.0,624.00,6431.00,98.8,1.8125,4.0,437.0,21.2,396.90,15.39,18.0
356,13,0.09378,12.5,7.87,0.0,524.00,5889.00,39.0,5.4509,5.0,311.0,15.2,390.50,15.71,21.7


### Registros duplicados

al revisar el dataset se encontró que varios registros (identificados por ID) están completamente duplicados. Esto no fue detectado en la primera versión de este notebook. Si no se eliminan, el dataset queda inflado artificialmente (la misma vivienda se cuenta más de una vez) y esto puede generar fuga de información (data leakage) más adelante, si por ejemplo la misma vivienda termina a la vez en el conjunto de entrenamiento y en el de prueba durante la etapa de modelado.

Se identifican y se eliminan los duplicados exactos, conservando solo la primera aparición de cada ID.

In [5]:
n_antes = boston_df.shape[0]
ids_duplicados = boston_df["ID"].duplicated().sum()
filas_duplicadas = boston_df.duplicated().sum()

print(f"Filas totales antes de limpiar: {n_antes}")
print(f"IDs duplicados: {ids_duplicados}")
print(f"Filas 100% duplicadas: {filas_duplicadas}")
print(f"Registros únicos (por ID): {boston_df['ID'].nunique()}")

Filas totales antes de limpiar: 448
IDs duplicados: 105
Filas 100% duplicadas: 80
Registros únicos (por ID): 343


In [6]:
boston_df = boston_df.drop_duplicates(subset="ID", keep="first").reset_index(drop=True)

print(f"Filas después de eliminar duplicados: {boston_df.shape[0]}")
boston_df.info()

Filas después de eliminar duplicados: 343
<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       343 non-null    int64  
 1   crim     339 non-null    float64
 2   zn       343 non-null    float64
 3   indus    339 non-null    float64
 4   chas     343 non-null    float64
 5   nox      341 non-null    float64
 6   rm       340 non-null    float64
 7   age      342 non-null    float64
 8   dis      340 non-null    float64
 9   rad      342 non-null    float64
 10  tax      340 non-null    float64
 11  ptratio  342 non-null    float64
 12  black    341 non-null    float64
 13  lstat    343 non-null    float64
 14  medv     331 non-null    float64
dtypes: float64(14), int64(1)
memory usage: 40.3 KB


### Valores nulos
En este conjunto de datos, los valores nulos se representan mediante las cadenas estándar `NaN` o posibles cadenas `?`, por lo que las reemplazamos/unificamos con `np.nan`.

In [7]:
boston_df = boston_df.replace("?", np.nan)
boston_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       343 non-null    int64  
 1   crim     339 non-null    float64
 2   zn       343 non-null    float64
 3   indus    339 non-null    float64
 4   chas     343 non-null    float64
 5   nox      341 non-null    float64
 6   rm       340 non-null    float64
 7   age      342 non-null    float64
 8   dis      340 non-null    float64
 9   rad      342 non-null    float64
 10  tax      340 non-null    float64
 11  ptratio  342 non-null    float64
 12  black    341 non-null    float64
 13  lstat    343 non-null    float64
 14  medv     331 non-null    float64
dtypes: float64(14), int64(1)
memory usage: 40.3 KB


Al revisar los rangos esperados de cada variable (según la
documentación original del Boston Housing dataset) se encontró que varias columnas numéricas
mezclan dos escalas distintas dentro de la misma columna, como si una parte de los registros
hubiera sido multiplicada por 1000 por error:

- `nox` debería estar entre 0 y ~1 (concentración como proporción), pero la mayoría de los valores
  están en cientos (ej. `871` en vez de `0.871`).
- `rm` debería estar entre ~3 y ~9 (habitaciones promedio), pero la mayoría de los valores están en
  miles (ej. `6998` en vez de `6.998`).
- `dis` debería estar entre ~1 y ~13, pero un subconjunto de valores está en miles (ej. `4233` en vez
  de `4.233`, el mismo valor ya observado como sospechoso en el notebook de entendimiento del problema).
- `crim` tiene 4 valores extremos también corridos de escala (ej. `15288` en vez de `15.288`).

Se corrigen dividiendo por 1000 únicamente los valores que superan un umbral físicamente imposible
para cada variable (se revisó que ningún valor legítimo de la columna cae en esa zona, para no alterar
datos correctos).

In [8]:
umbrales_escala = {
    "nox": 10,  # nox real está entre 0 y ~1
    "rm": 15,  # rm real está entre ~3 y ~9
    "dis": 15,  # dis real está entre ~1 y ~13
    "crim": 100,  # crim real llega hasta ~74 en este dataset
}

for col, umbral in umbrales_escala.items():
    mask = boston_df[col] > umbral
    n_corregidos = mask.sum()
    boston_df.loc[mask, col] = boston_df.loc[mask, col] / 1000
    print(f"{col}: {n_corregidos} valores corregidos (divididos entre 1000)")

print()
for col in umbrales_escala:
    print(f"{col}: min={boston_df[col].min():.3f}  max={boston_df[col].max():.3f}")

nox: 288 valores corregidos (divididos entre 1000)
rm: 306 valores corregidos (divididos entre 1000)
dis: 33 valores corregidos (divididos entre 1000)
crim: 4 valores corregidos (divididos entre 1000)

nox: min=0.385  max=0.871
rm: min=3.561  max=8.725
dis: min=1.130  max=10.710
crim: min=0.006  max=73.534


### Eliminar columnas
Eliminaremos la columna `ID` porque es solo un identificador de registro y no contiene información predictiva para la variable objetivo `medv`.

In [9]:
boston_df = boston_df.drop(columns=["ID"])
boston_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     339 non-null    float64
 1   zn       343 non-null    float64
 2   indus    339 non-null    float64
 3   chas     343 non-null    float64
 4   nox      341 non-null    float64
 5   rm       340 non-null    float64
 6   age      342 non-null    float64
 7   dis      340 non-null    float64
 8   rad      342 non-null    float64
 9   tax      340 non-null    float64
 10  ptratio  342 non-null    float64
 11  black    341 non-null    float64
 12  lstat    343 non-null    float64
 13  medv     331 non-null    float64
dtypes: float64(14)
memory usage: 37.6 KB


### Variables categóricas

#### Nominal
- `chas`: Variable ficticia del río Charles (1 si el área limita con el río; 0 en caso contrario)

#### Ordinal
- `rad`: Índice de accesibilidad a las carreteras radiales

### Variables numéricas

#### Variables continuas
- `crim`: Tasa de criminalidad per cápita por municipio
- `zn`: Proporción de terrenos residenciales zonificados para lotes de más de 25 000 pies cuadrados
- `indus`: Proporción de hectáreas destinadas a negocios no minoristas por municipio
- `nox`: Concentración de óxidos de nitrógeno (partes por 10 millones)
- `rm`: Número promedio de habitaciones por vivienda
- `age`: Proporción de viviendas ocupadas por sus propietarios construidas antes de 1940
- `dis`: Distancias ponderadas a cinco centros de empleo de Boston
- `tax`: Tasa del impuesto predial sobre el valor total por cada $10,000
- `ptratio`: Proporción alumno-profesor por municipio
- `black`: 1000(Bk - 0.63)^2, donde Bk es la proporción de personas negras por municipio
- `lstat`: Porcentaje de población con estatus socioeconómico bajo
- `medv`: Valor medio de las viviendas ocupadas por sus propietarios en miles de dólares (variable objetivo)

## Convertir tipos de datos

### Variables categóricas

In [10]:
cols_categoric = ["chas", "rad"]
boston_df[cols_categoric] = boston_df[cols_categoric].astype("category")

### Variables numéricas

In [11]:
cols_numeric_float = [
    "crim",
    "zn",
    "indus",
    "nox",
    "rm",
    "age",
    "dis",
    "tax",
    "ptratio",
    "black",
    "lstat",
    "medv",
]

boston_df[cols_numeric_float] = boston_df[cols_numeric_float].astype("float64")

boston_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   crim     339 non-null    float64 
 1   zn       343 non-null    float64 
 2   indus    339 non-null    float64 
 3   chas     343 non-null    category
 4   nox      341 non-null    float64 
 5   rm       340 non-null    float64 
 6   age      342 non-null    float64 
 7   dis      340 non-null    float64 
 8   rad      342 non-null    category
 9   tax      340 non-null    float64 
 10  ptratio  342 non-null    float64 
 11  black    341 non-null    float64 
 12  lstat    343 non-null    float64 
 13  medv     331 non-null    float64 
dtypes: category(2), float64(12)
memory usage: 33.0 KB


## 💾 Guardar dataframe 

In [12]:
# Subir hasta la raíz del proyecto (Precios-casas-Boston)
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

output_dir = DATA_DIR / "02_intermediate"
output_dir.mkdir(parents=True, exist_ok=True)

# Convertir y guardar (ya con duplicados eliminados y escalas corregidas)
table = pa.Table.from_pandas(boston_df, preserve_index=False)
pq.write_table(table, output_dir / "boston_type_fixed.parquet")

print("Guardado en la ubicación correcta:", output_dir / "boston_type_fixed.parquet")

Guardado en la ubicación correcta: /home/maricel98/Precios-casas-Boston/data/02_intermediate/boston_type_fixed.parquet


## 📊 Análisis de resultados
Se realizaron las siguientes correcciones sobre el dataset RAW:

- Se identificaron y eliminaron registros duplicados (mismo `ID` repetido), quedando únicamente
  observaciones únicas por vivienda.
- Se identificaron y corrigieron inconsistencias de escala dentro de las columnas `nox`, `rm`, `dis`
  y `crim`, donde un subconjunto de valores estaba multiplicado por 1000 respecto al resto de la
  columna, para que cada variable tenga una unidad y un tipo de dato uniforme.
- Se eliminó la columna `ID`, que solo funciona como identificador y no aporta información predictiva.
- Se corrigieron los tipos de dato a las categorías de pandas y `float64` correctas (`chas` y `rad`
  como `category`, el resto como `float64`).
- Los valores nulos se unificaron con `np.nan` para permitir un análisis y visualización correctos
  de los datos.

Aún quedan pendientes para la siguiente etapa (preparación de datos): decidir qué hacer con los
registros que tienen una proporción muy alta de valores faltantes, y con los 16 registros que no
tienen valor en `medv` (la variable objetivo), ya que estos últimos no pueden usarse para entrenar
un modelo supervisado.

## 💡 Propuestas e ideas
- Definir una estrategia de imputación para los valores faltantes restantes antes del desarrollo del modelo.
- Decidir si los registros sin `medv` se descartan o se reservan para un uso distinto (no se pueden usar para entrenar).
- Evaluar el impacto de haber eliminado los duplicados sobre la distribución de las variables (comparar estadísticas antes/después).
- Considerar el backend PyArrow para optimizar el uso de memoria.
- Realizar el split de entrenamiento/prueba en la siguiente etapa, ya sobre el dataset sin duplicados, para evitar que la misma vivienda quede tanto en train como en test.

## 📖 Referencias
- https://pandas.pydata.org/docs/user_guide/pyarrow.html
- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/